```{ojs}
config = ({
  view: {stroke: null}, 
  background: "transparent",
  padding: {
    left: 0, 
    top: 5, 
    right: 0, 
    bottom: 5,
    },
  font: "Montserrat",
  fontSize: 14, 
  title: {
    offset: 0, 
    fontSize: 16, 
    subtitleFontSize: 14,
  },
  facet: {
    fontSize: 14,
    labelFontSize: 14, 
    titleFontSize: 14,
    titleFontWeight: "normal",
  },
  axis: {
    labelFontSize: 14, 
    titleFontSize: 14,
    titleFontWeight: "normal",
  },
  legend: {
    labelFontSize: 14, 
    titleFontSize: 14,
    titleFontWeight: "normal",
  },
  mark: {
    fontSize: 14
  },
  locale: {
    number: {
      decimal: ",", 
      thousands: ".", 
      grouping: [3]
    },
    time: {
      dateTime: "%A %e %B %Y, %X",
      date: "%d/%m/%Y",
      time: "%H:%M:%S",
      periods: ["AM", "PM"],
      days: [
        "Montag",
        "Dienstag",
        "Mittwoch",
        "Donnerstag",
        "Freitag",
        "Samstag",
        "Sonntag"
      ],
      shortDays: [
        "Mo", 
        "Di", 
        "Mi", 
        "Do", 
        "Fr", 
        "Sa", 
        "So"
      ],
      months: [
        "Jänner",
        "Februar",
        "März",
        "April",
        "Mai",
        "Juni",
        "Juli",
        "August",
        "September",
        "Oktober",
        "November",
        "Dezember"
      ],
      shortMonths: [
        "Jan",
        "Feb",
        "Mar",
        "Apr",
        "Mai",
        "Jun",
        "Jul",
        "Aug",
        "Sep",
        "Okt",
        "Nov",
        "Dez"
      ]
    }
  },
})
```


```{ojs}
colors2 = ["#be0021", "#ffa500"] // ["#be0021", "#ffa500"]
colors = ["#d3d3d3", "#0063b1"] // ["#0063b1", "#008000"] 
colors3 = ["#4b0082", "#00ced1"]
colors4 = ["#008080", "#800080"]
colors5 = ["#00ff00", "#0000ff"]

function makeWidth(inputWidth) {
  const outputWidth = 
    inputWidth > 2000 ? inputWidth * 0.250 :
    inputWidth > 1600 ? inputWidth * 0.330 :
    inputWidth > 1400 ? inputWidth * 0.380 :
    inputWidth > 1100 ? inputWidth * 0.400 :
    inputWidth > 825 ? inputWidth * 0.450 :
    inputWidth * 0.775
  return outputWidth
}

germanLocale = d3.formatLocale({
  decimal: ",", // Decimal separator
  thousands: ".", // Thousands separator
  grouping: [3],  // Grouping for thousands
  currency: ["€", ""],  // Currency format (optional)
});

// function makeWidthSmall(inputWidth) {
//   const outputWidth = 
//     inputWidth > 2000 ? inputWidth * 0.080 : // Home and or WIFO screen?
//     inputWidth > 1600 ? inputWidth * 0.100 : // ASCII screen (small)
//     inputWidth > 1400 ? inputWidth * 0.150 : // laptop
//     inputWidth > 1100 ? inputWidth * 0.166 : // 
//     inputWidth > 825 ? inputWidth * 0.200 : // 
//     inputWidth * 0.380
//   return outputWidth
// }

// function makeWidthBars(inputWidth) {
//   const outputWidth = 
//     inputWidth > 2000 ? inputWidth * 0.100 : // Home and or WIFO screen?
//     inputWidth > 1600 ? inputWidth * 0.125 : // ASCII screen (small)
//     inputWidth > 1400 ? inputWidth * 0.150 : // laptop
//     inputWidth > 1100 ? inputWidth * 0.200 : // 
//     inputWidth > 825 ? inputWidth * 0.300 : // 
//     inputWidth * 0.8
//   return outputWidth
// }

logoAK = vl
  .markImage({
    x: makeWidth(width)/1.7 - 48,
    x2: makeWidth(width)/1.7,
    y: 20,
    opacity: 0.2,
    clip: true,
    smooth: true,
    height: 100,
  })
  .data([
    {x: 1, y: 2}
  ])
  .encode(
    vl.url()
      .value("https://appetizingdata.com/wp-content/uploads/2024/08/ak-logo.svg"),
  )

logoWIFO = vl
  .markImage({
    x: makeWidth(width)/1.7 - 48,
    x2: makeWidth(width)/1.7 - 2,
    y: 50,
    opacity: 0.2,
    clip: true,
    smooth: true,
    height: 100,
  })
  .data([
    {x: 1, y: 2}
  ])
  .encode(
    vl.url()
      .value("https://appetizingdata.com/wp-content/uploads/2024/08/WIFO_kurz_rechts.svg"),
  )

logo = vl.layer(logoAK, logoWIFO)
```


<link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.6.0/css/all.min.css">



```{ojs}
// echo: false
// warning: false
import {vl} from "@vega/vega-lite-api-v5" // apiVersion = '5.6.0' vlVersion = '5.6.0' vegaVersion = '5.23.0' tooltipVersion = '0.30.0'
import {aq, op} from '@uwdata/arquero' // version "6.0.0"
// import {number, slider} from "@jashkenas/inputs"
// import { rangeInput } from '@mootari/range-slider'
```

```{ojs}
descriptions = FileAttachment("data/descriptions.csv").csv({ typed: true }) 
```

```{ojs}
binsRaw = FileAttachment("data/bins.csv").csv({ typed: true })
scenarios = aq.from(binsRaw)
  .derive({ variable: d => op.replace(d.variable, /\r\n/g, " ") })
  .filter(aq.escape(d => d.variable != "Status quo"))
  .filter(aq.escape(d => d.variable != "Szenario 4 (GPG/Frau)"))
  .filter(aq.escape(d => d.variable != "Szenario 5 (Beste 15 Jahre)"))
  .select("variable")
  .dedupe()
  .array("variable")
```

```{ojs}
info = descriptions
  .filter(d => d.variable == select)
  .map(d => d.description)[0] 
```

```{ojs}
binsMid = aq.from(binsRaw)
  .derive({ 
    variable: d => op.replace(d.variable, /\r\n/g, " "),
    bin: d => d.bin * 100,
    range: aq.escape(d => range),
    radio: aq.escape(d => radio),
  })

cumPercMid = binsMid
  .derive({
    tail: aq.escape(
      d => d.bin < range ? "lower" : 
      d.bin > range ? "higher" : "same"
      )
  })
  .groupby(["gesl", "variable", "tail"]) 
  .rollup({ N: d => op.sum(d.N) })
  .groupby(["gesl", "variable"])
  .derive({ perc: d => d.N / op.sum(d.N) })

CumPercSame = cumPercMid
  .filter(d => d.tail == "same")
  .derive({
    percSame: d => d.perc,
  })
  .select(["gesl", "variable", "percSame"],)

cumPerc = cumPercMid
  .join_left(CumPercSame, [["gesl", "variable"], ["gesl", "variable"]])
  .derive({
    perc: d => d.tail == "lower" ? d.perc + d.percSame : d.perc, // closest decile to the right! not left!!!!
  })
  .groupby(["gesl", "variable", "tail"])
  .derive({ text: aq.escape(d => germanLocale.format(",.0~%")(d.perc)) })
  .objects()

function getCumPerc(gesl, tail) {
  const out = cumPerc
  .filter(d => 
    d.gesl == gesl &&
    d.tail == tail && 
    d.variable == "Status quo"
    )
  .map(d => d.text)
  return out
}

notRadio = radio == "Frauen" ? "Männer" : "Frauen"
```

```{ojs}
decilesRaw = 
  check1[0] == "Kindererziehungszeiten" ?
  FileAttachment("data/deciles_betroffen_szen1.csv").csv({ typed: true }) :
  check2[0] == "Arbeitslosenversicherungszeiten" ?
  FileAttachment("data/deciles_betroffen_szen2.csv").csv({ typed: true }) :
  FileAttachment("data/deciles.csv").csv({ typed: true })

decilesMid = aq.from(decilesRaw)
  .derive({ 
    select: aq.escape(d => select),
    range: aq.escape(d => range),
    radio: aq.escape(d => radio),
    diff: aq.escape(d => Math.abs(d.value-range)), // closest decile to the right! not left!!!!
  }) 
  .groupby(["gesl", "val", "variable"])
  .derive({ closestDecile: d => op.min(d.diff) == d.diff && d.gesl == d.radio && d.val == "Status quo" ? 1 : 0 })
  .groupby(["gesl", "variable", "D"])
  .derive({ closestDecile: d => op.sum(d.closestDecile) > 0 ? "yes" : "no" })
  .ungroup()

decilesDiffVal = decilesMid
  .filter(aq.escape(d => d.variable == select))
  .select(["gesl", "D", "variable", "val", "value"])
  .groupby(["gesl", "D", "variable"])
  .pivot("val", "value")
  .derive({ 
    diffScenario: d => d["Szenario"] - d["Status quo"],
    diffScenarioPerc: d => (d["Szenario"] - d["Status quo"]) / d["Status quo"],
  })

closestDecileValue = decilesMid
  .filter(aq.escape(d => d.variable == select))
  .objects()
  .filter(d => d.closestDecile == "yes")
  .map(d => d.D)[0]

decilesDiffGesl = decilesMid
  .derive({ closestDecileValue: aq.escape(d => closestDecileValue) })
  .select(["gesl", "D", "variable", "val", "value"])
  .groupby(["val", "D", "variable"])
  .pivot("gesl", "value")
  .derive({ 
    gapValue: d => d.Männer - d.Frauen,
    gapPerc: d => (d.Männer - d.Frauen) / (d.Männer),
    gapRel: d => (d.Frauen) / (d.Männer),
  })

deciles = decilesMid
  .derive({ closestDecileValue: aq.escape(d => closestDecileValue) })
  .join_left(decilesDiffVal, [["gesl", "D", "variable"], ["gesl", "D", "variable"]])
  .join_left(decilesDiffGesl, [["val", "D", "variable"], ["val", "D", "variable"]])
  .objects()

medians = decilesMid
  .filter(aq.escape(d => d.D == "50. Perzentil"))
  .derive({ 
    variable: d => d.val == "Szenario" ? d.variable : "Status quo",
    median: d => d.value,
    })
  .select(["gesl", "variable", "val", "median"])
  .dedupe()

bins = binsMid
  .join_left(medians, [["gesl", "variable"], ["gesl", "variable"]])
  .objects()

```



<!-- kommunikation@akooe.at [undefined:kommunikation@akooe.at] -->

<div class="image-container"> <img src="img/header.jpeg"> </div>

::: {.column-body .mainText .scroll-target}
<h4 class="headlineRechner">RECHNER</h4>
<!-- # AK-Pensionsreformrechner  -->
:::

:::: {#slideshow-container .column-body}

::: {.slide .slide-active}

## Pensionsreformen: AK-Vorschläge für eine faire Berechnung

#### Einstiegsfrage

**Um wieviel Prozent ist die durchschnittliche Pension von Frauen in Österreich geringer als die der Männer?** 

<div style="width: 500px;">


```{ojs}
viewof guess = Inputs.radio(["0%", "18,9% ", "40,1%", "5,7%"], {
  label: "", value: "0%", // Wählen Sie eine Antwort
})
// guess // is null
answer = guess == "40,1%" ? "Richtig! Der Unterschied ist groß." : "Leider falsch, die Realität sieht schlimmer aus."
```


</div> 

:::

::: {.slide}

## Pensionsreformen: AK-Vorschläge für eine faire Berechnung

#### Auflösung

**${answer}**

Der Gender Pension Gap, also der Unterschied zwischen Männer- und Frauenpensionen, liegt im Jahr 2024 in Österreich bei 40,1%¹. In Oberösterreich liegt er sogar bei 45,4% und ist somit im Bundesländervergleich nach Vorarlberg an vorletzter Stelle. Gründe für die Schlechterstellung von Frauen sind die Hürden, mit denen sie bereits im Erwerbsleben konfrontiert sind: lange Teilzeitphasen, niedrigere Einkommen, Zeiten von Arbeitslosigkeit und Unterbrechungen aufgrund von Kindererziehungs- und Pflegezeiten.

Die Arbeiterkammer Oberösterreich hält dies für eine große Ungerechtigkeit und setzt sich für Pensionsreformen ein, die tatsächlich Verbesserungen für die Menschen bringen und dieser enormen Schieflage entgegenwirken.

Mit dem AK-Pensionsreformrechner sehen Sie, was die einzelnen Reformen für die Pensionshöhen, bei der Reduktion des Gender Pension Gap und Verteilung bedeuten. Probieren Sie das neue Tool aus, verbreiten Sie es und geben Sie uns Rückmeldung!

### Machen Sie mit! Wie viel Pension würden Sie bekommen?

Eine [Studie des WIFO](res/AKOE_Pensionen_EB_070824.pdf){target="_blank"} 
im Auftrag der AK OÖ untersucht die Auswirkungen verschiedener Änderungen im Pensionsrecht auf die Höhe der Alterseinkommen und den Gender
Pension Gap.²

<ul>
  <li>Szenario 1: Höherbewertung von Kindererziehungszeiten</li>
  <li>Szenario 2: Höherbewertung von Zeiten der Arbeitslosigkeit</li>
  <li>Szenario 3: Einführung eines Gender-Pay-Gap-Faktors, der die Einkommensunterschiede zwischen Männern und Frauen ausgleicht.</li>
</ul>

Die Auswirkungen dieser Szenarien auf Pensionen und Haushaltseinkommen können Sie hier erkunden.

:::{.infoText}
¹Datenbasis: Pensionsversicherungs-Jahresstatistik, Dezember 2023, Berechnung: MA 23 - Wirtschaft, Arbeit und Statistik. <br>
²Datenbasis: Sozialversicherungsdaten zu und den Erwerbs- und Einkommensverläufen der Pensionsantrittskohorten 2015 bis 2021. 
:::

:::

::: {.slide}
## Geben Sie eine Bruttopension an
#### Auswahl 1/3
<!-- <span>Geben Sie eine Bruttopension an</span> -->

```{ojs}
viewof rangeRaw = Inputs.range([400, 4000], {step: 50, value: 1500})
range = Math.round(rangeRaw/100)*100
```


:::{.infoText}
![](img/Infobox_info_icon.svg){height=14px} 
Sollten Sie ihre voraussichtliche Bruttopension nicht kennen können Sie diese mit dem [AK-Pensionsrechner](https://pensionsrechner.arbeiterkammer.at/pkr-on2/){target="_blank"} ermitteln. 
<br>
![](img/Infobox_info_icon.svg){height=14px} 
Ihre Bruttopension wird in weiterer Folge dazu verwendet Sie in eine Vergleichsgruppe einzuordnen.
:::

<!-- **Auswahl 2/4**: 
Geben Sie Ihr Geschlecht an

```{ojs}
viewof radioRaw = Inputs.radio(["Frau", "Mann"], {value: "Frauen"})
```

![](img/Infobox_info_icon.svg){height=14px}
Manche der berechneten Szenarien wirken sich nur auf Frauen aus. Daher ist es möglich, dass die Ergebnisse bei Männern keine Veränderungen zeigen. -->

:::

::: {.slide}
## Geben Sie ein Geschlecht an
#### Auswahl 2/3
<!-- <span>Geben Sie ein Geschlecht an</span> -->

```{ojs}
viewof radioRaw = Inputs.radio(["Frau", "Mann"], {value: "Frau"})
radio = radioRaw == "Frau" ? "Frauen" : "Männer"
```


:::{.infoText}
![](img/Infobox_info_icon.svg){height=14px}
Manche der berechneten Szenarien wirken sich nur auf Frauen aus. Daher ist es möglich, dass die Ergebnisse bei Männern keine Veränderungen zeigen. <br>
<!-- ![](img/Infobox_info_icon.svg){height=14px}
Datenbasis der Kohorten basiert auf einer einen nicht-binären Geschlechtseintrag noch nicht gegeben hat -->
:::

:::


::: {.slide}
## Wählen Sie ein Szenario
#### Auswahl 3/3
<!-- <span>Wählen Sie ein Szenario</span> -->

```{ojs}
viewof select = Inputs.select(scenarios)
```


:::{.infoText}
![](img/Infobox_info_icon.svg){height=14px}
Informationen zu dem von Ihnen gewählten Szenario: ${info} <br>
:::

<span>Zutreffendes ankreuzen</span>


```{ojs}
viewof check1 = Inputs.checkbox(["Kindererziehungszeiten"], {disabled: select == "Szenario 1 (Höherbewertung Kindererziehungszeiten)" && radio == "Frauen" ? false : true})
viewof check2 = Inputs.checkbox(["Arbeitslosenversicherungszeiten"], {disabled: select == "Szenario 2 (Höherbewertung Zeiten der Arbeitslosigkeit)" ? false : true})
interpolate = "step"
check1Info = select != "Szenario 1 (Höherbewertung Kindererziehungszeiten)" ? "Kindererziehungszeiten sind nur im Szenario 1 relevant." : radio != "Frauen" ? "Kindererziehungszeiten wurden bei der Berechnung der Szenarien nur für Frauen berücksichtigt." : "Kindererziehungszeiten werden für Frauen in durchschnittlichem Ausmaß angenommen."
check2Info = select != "Szenario 2 (Höherbewertung Zeiten der Arbeitslosigkeit)" ? "Arbeitslosenversicherungsszeiten sind nur im Szenario 2 relevant." : "Es werden Arbeitslosenversicherungszeiten von durchschnittlicher Höhe angenommen."
```


:::{.infoText}
![](img/Infobox_info_icon.svg){height=14px} ${check1Info} <br> ![](img/Infobox_info_icon.svg){height=14px} ${check2Info}
:::

:::

<!-- ::: {.slide}
**Auswahl 4/4**: 
Zutreffendes ankreuzen (falls relevant)

```{ojs}
viewof check1 = Inputs.checkbox(["Kindererziehungszeiten"], {disabled: select == "Szenario 1 (Höherbewertung Kindererziehungszeiten)" && radio == "Frauen" ? false : true})
viewof check2 = Inputs.checkbox(["Arbeitslosenversicherungszeiten"], {disabled: select == "Szenario 2 (Höherbewertung Zeiten der Arbeitslosigkeit)" ? false : true})
interpolate = "step"
check1Info = select != "Szenario 1 (Höherbewertung Kindererziehungszeiten)" ? "Kindererziehungszeiten wären nur im Szenario 1 relevant." : radio != "Frauen" ? "Kindererziehungszeiten wurden bei der Berechnung der Szenarien nur für Frauen berücksichtigt." : "Kindererziehungszeiten werden für Frauen in durchschnittlichem Ausmaß angenommen."
check2Info = select != "Szenario 2 (Höherbewertung Zeiten der Arbeitslosigkeit)" ? "Arbeitslosenversicherungsszeiten wären nur im Szenario 2 relevant." : "Es werden Arbeitslosenversicherungszeiten von durchschnittlicher Höhe angenommen."
```

![](img/Infobox_info_icon.svg){height=14px} ${check1Info} <br> ![](img/Infobox_info_icon.svg){height=14px} ${check2Info}
::: -->

::: {.slide}
## Das sind Ihre Ergebnisse
#### Ergebnisse 1/4
<span>Wie würde sich das ${select} auf vergleichbare Personen auswirken?</span>

```{ojs}
decilesBar = 
  deciles
    .filter(d => d.variable == select)
    .filter(d => d.D == d.closestDecileValue || d.D == "Gesamt")
    .filter(d => d.gesl == d.radio || d.D == "Gesamt")
    .map(d => ({...d, D: d.D == "Gesamt" ? "∅ Gesamtdurchschnitt" : "Meine Vergleichsgruppe"}))
    .map(d => ({...d, gesl: d.gesl == d.radio && d.D == "Meine Vergleichsgruppe" ? "Meine Vergleichsgruppe" : "∅ " + d.gesl}))
    .map(d => ({...d, dummy: "⠀"}))

function bars(inputWidth) {

  const bar = vl
    .markBar({
      strokeWidth: 1.75,
      strokeOpacity: 1,
      // size: width > 1600 ? 20 : null,
    })
    .encode(
      vl.x()
        .fieldN("gesl")
        .sort("value")
        .axis({
          minExtent: 70,
          grid: false,
          title: null,
        }), 
      vl.y()
        .fieldQ("value")
        .scale({
          domain: [0, 5000],
          nice: false,
        })
        .axis({
          domain: false,
          grid: false,
          title: width > 600 ? "Bruttopension" : null, 
          labels:  width > 600 ? true : false,
          ticks:  width > 600 ? true : false,
        }),
      vl.xOffset()
        .fieldN("val"),
      vl.color()
        .fieldN("val")
        .scale({
          range: colors,
        })
        .legend({
          // orient: width > 825 ? "right" : "top",
          orient: "top",
          symbolOpacity: 1,
          symbolType: "square",
          title: null,
        }),
      vl.stroke()
        .fieldN("closestDecile")
        .scale({
          range: radio2 == "Meine Vergleichsgruppe" ? 
            ["transparent", "black"] : 
            radio2 == "∅ Gesamtdurchschnitt" ? 
            ["black", "transparent"] :
            ["black", "black"]          
        })
        .legend(null),
      vl.opacity()
        .fieldN("closestDecile")
        .scale({
          range: radio2 == "Meine Vergleichsgruppe" ? 
            [0.5, 0.8] : 
            radio2 == "∅ Gesamtdurchschnitt" ? 
            [0.8, 0.5] :
            [0.8, 0.8]
        })
        .legend(null),
    )

  const text = vl
    .markText({
      clip: true,
      size: 14,
      y: width > 600 ? 270 : 230,
      angle: -90,
      align: "left", 
    })
    .encode(
      vl.x()
        .fieldO("gesl"),
      vl.xOffset()
        .fieldN("val"),
      vl.text()
        .fieldQ("value")
        .format(",.0~f"),
      vl.opacity()
        .fieldN("closestDecile")
        .scale({
          range: radio2 == "Meine Vergleichsgruppe" ? 
            [0.5, 0.8] : 
            radio2 == "∅ Gesamtdurchschnitt" ? 
            [0.8, 0.5] :
            [0.8, 0.8]
        })
        .legend(null),
    )

  const textDiff = vl
    .markText({
      clip: true,
      size: 14,
      dy: -16,
      align: "center",
    })
    .transform(vl.filter("datum.val == 'Szenario'"))
    .encode(
      vl.x()
        .fieldO("gesl"),
      vl.y()
        .fieldQ("value"),
      vl.text()
        .fieldQ("diffScenario")
        .format("+,.0~f"),
      vl.opacity()
        .fieldN("closestDecile")
        .scale({
          range: radio2 == "Meine Vergleichsgruppe" ? 
            [0.5, 0.8] : 
            radio2 == "∅ Gesamtdurchschnitt" ? 
            [0.8, 0.5] :
            [0.8, 0.8]
        })
        .legend(null),
    )

  const textDiffPerc = vl
    .markText({
      clip: true,
      size: 14,
      dy: -36,
      align: "center",
    })
    .transform(vl.filter("datum.val == 'Szenario'"))
    .encode(
      vl.x()
        .fieldO("gesl"),
      vl.y()
        .fieldQ("value"),
      vl.text() 
        .fieldQ("diffScenarioPerc")
        .format("+,.0~%"),
      vl.opacity()
        .fieldN("closestDecile")
        .scale({
          range: radio2 == "Meine Vergleichsgruppe" ? 
            [0.5, 0.8] : 
            radio2 == "∅ Gesamtdurchschnitt" ? 
            [0.8, 0.5] :
            [0.8, 0.8]
        })
        .legend(null), 
    )

  const textPosition = vl
    .markText({
      clip: true,
      size: 18,
      dy: -10,
      dx: -28,
      // angle: -0,
      align: "left",
      strokeWidth: 1.25,
    })
    .transform(
      vl.filter("datum.D == 'Vergleichsgruppe'"),
      vl.filter("datum.val == 'Status quo'"), 
    )
    .encode(
      vl.x()
        .fieldN("gesl")
        .sort("value"), 
      vl.y()
        .fieldQ("value"),
      vl.text()
        .value("↘"),
      vl.opacity()
        .fieldN("closestDecile")
        .scale({
          range: radio2 == "Meine Vergleichsgruppe" ? 
            [0.5, 0.8] : 
            radio2 == "∅ Gesamtdurchschnitt" ? 
            [0.8, 0.5] :
            [0.8, 0.8]
        })
        .legend(null),  
    )

  const layers = width > 600 ?
    vl.layer(
      bar,
      text,
      textDiff,
      textDiffPerc,
      textPosition,
    ) :
    vl.layer(
      bar,
      text,
      textDiff,
      textDiffPerc,
    ) 

  return layers
    .height(width > 600 ? 280 : 240) // width > 825 ? 280 : 180 
    .width(inputWidth)
    .facet({
      field: "dummy", 
      type: "nominal", 
      title: null,
      header: { 
        labelFontSize: 14, 
        orient: "top",
      },
    })
    // .resolve({
    //   "axis": {"x": "independent"},
    //   "scale": {"x": "independent"},
    //   })
    .data(decilesBar)
    .config(config)

}

bars(width > 600 ? 250 : width * 0.8).render({ renderer: "svg" })

Inputs.bind(
  Inputs.radio(
    ["Meine Vergleichsgruppe", "∅ Gesamtdurchschnitt", "Beides anzeigen"], 
    {value: "Meine Vergleichsgruppe", label: "Zeige"}
    ),
  viewof radio2
  )
```

```{ojs}
checkText1 = check1.length == 1 ? ", die Kinderbetreuungszeiten vorweisen können," : ""
checkText2 = check2.length == 1 ? ", die Ersatzzeiten durch Arbeistlosigkeit haben," : ""

diffScenario = deciles
  .filter(d => d.variable == select)
  .filter(d => d.gesl == radio && d.val == "Szenario" && d.D == d.closestDecileValue)
  .map(d => d.diffScenario)
diffScenarioGesamt = deciles
  .filter(d => d.variable == select)
  .filter(d => d.gesl == radio && d.val == "Szenario" && d.D == "Gesamt")
  .map(d => d.diffScenario)
diffScenarioPerc = deciles
  .filter(d => d.variable == select)
  .filter(d => d.gesl == radio && d.val == "Szenario" && d.D == d.closestDecileValue)
  .map(d => d.diffScenarioPerc)
diffScenarioPercGesamt = deciles
  .filter(d => d.variable == select)
  .filter(d => d.gesl == radio && d.val == "Szenario" && d.D == "Gesamt")
  .map(d => d.diffScenarioPerc)
statusIncome = deciles
  .filter(d => d.closestDecileValue == d.D && d.gesl == d.radio && d.val == "Status quo")
  .map(d => d.value)[0]

barsText = 
  diffScenario > 0 && diffScenarioGesamt > 0 ? 
  "Im " + select + " würden vergleichbere " + radio + " mit einer Pension von " + germanLocale.format(",.0f")(statusIncome) + " Euro (" + deciles.map(d => d.closestDecileValue)[0] + " der " + radio + ")" + checkText1 + checkText2 + " im Schnitt um " + germanLocale.format(",.0f")(diffScenario) + " Euro mehr Pension erhalten. Das entspricht etwa " + germanLocale.format(",.0%")(diffScenarioPerc) + " mehr. Betrachtet man den Gesamtdurchschnitt aller "+ radio + checkText1 + checkText2 + " so würden diese im Schnitt " + germanLocale.format(",.0f")(diffScenarioGesamt) + " Euro oder " + germanLocale.format(",.0%")(diffScenarioPercGesamt) + " mehr Pension bekommen." :
  diffScenarioGesamt < 0 ?
  "Im " + select + " würden vergleichbere " + radio + " mit einer Pension von " + germanLocale.format(",.0f")(statusIncome) + " Euro (" + deciles.map(d => d.closestDecileValue)[0] + " der " + radio + ")" + checkText1 + checkText2 + " im Schnitt um " + germanLocale.format(",.0f")(diffScenario) + " Euro mehr Pension erhalten. Das entspricht etwa " + germanLocale.format(",.0%")(diffScenarioPerc) + " mehr. Betrachtet man den Gesamtdurchschnitt aller "+ radio + checkText1 + checkText2 + " so würden diese im Schnitt " + germanLocale.format(",.0f")(-diffScenarioGesamt) + " Euro oder " + germanLocale.format(",.0%")(-diffScenarioPercGesamt) + " weniger Pension bekommen." : 
  "Im " + select + " würden " + radio + " mit einer Pension von " + germanLocale.format(",.0f")(statusIncome) + " Euro" + checkText2 + " gleich viel erhalten wie aktuell. " + "Das Szenario hätte keinen Effekt. " 
```


${barsText} <br><br> Eingaben ändern:


```{ojs}
Inputs.bind(Inputs.range([400, 4000], {step: 50, value: 1500, label: "Bruttopension"}), viewof rangeRaw)
Inputs.bind(Inputs.radio(["Frau", "Mann"], {value: "Frauen", label: "Geschlecht"}), viewof radioRaw)
Inputs.bind(Inputs.select(scenarios, {label: "Szenario"}), viewof select)
```


:::

::: {.slide}
## Das sind Ihre Ergebnisse
#### Ergebnisse 2/4
<span>Wie würde sich das ${select} auf den Gender Pension Gap auswirken?</span>


```{ojs}
decilesDumbbellFull = deciles
    .filter(d => (d.D == d.closestDecileValue || d.D == "Gesamt") && d.variable == select)
    .map(d => ({...d, D: d.D == "Gesamt" ? "∅ Gesamtdurchschnitt" : "Meine Vergleichsgruppe"}))
```

```{ojs}
function dumbbell(inputData, inputWidth)  {

  const points = vl
    .markPoint({
      size: 100,
      opacity: 1,
      shape: "square",
      stroke: "transparent",
    })
    .encode(
      vl.y()
        .fieldQ("value")
        .axis({
          title: width > 600 ? "Bruttopension" : null,
          labels:  width > 600 ? true : false,
          ticks:  width > 600 ? true : false,
          domain: false,
          grid: false,
        })
        .scale({
          domain: [0, 5000],
          nice: false,
        }),
      vl.x()
        .fieldO("val")
        .axis({ 
          title: null, 
        }),
      vl.fill()
        .fieldN("gesl")
        .scale({
          range: colors2,
        })
        .legend({
          // orient: width > 825 ? "right" : "top", 
          orient: "top",
          symbolOpacity: 1,
          symbolType: "square",
          title: null, 
        }),
    )

  const lines = vl
    .markLine({
      opacity: 0.15,
      size: 5,
    })
    .encode(
      vl.y()
        .fieldQ("value"),
      vl.x()
        .fieldO("val"),
      vl.color()
        .fieldN("val")
        .scale({
          range: ["black", "black"]
        })
        .legend(null),
    )

  const text = vl
    .markText({
      size: 14,
      dx: 27,
      clip: true,
    })
    .encode(
      vl.y()
        .fieldQ("value"),
      vl.x() 
        .fieldO("val"),
      vl.color()
        .fieldN("gesl")
        .scale({
          range: ["black"],
        }) 
        .legend(null),
      vl.text() 
        .fieldQ("value")
        .format(",.0f"), 
    )

  const textRel = vl
    .markText({
      size: 14,
      dx: width > 600 ? 27 : 10,
      dy: 18,
      clip: true,
    })
    .encode(
      vl.y()
        .fieldQ("value"),
      vl.x()
        .fieldO("val"),
      vl.text()
        .fieldQ("gapRel")
        .format(",.0~%"),
      vl.opacity()
        .fieldN("gesl")
        .scale({
          range : [0.51, 0] 
        })
        .legend(null), 
    )

  const textGap = vl
    .markText({
      size: 14,
      dx: 24,
      dy: -16,
      angle: -90,
      align: "right",
      clip: true,
    })
    .encode(
      vl.y()
        .fieldQ("value"),
      vl.x()
        .fieldO("val"),
      vl.text()
        .fieldQ("gapValue")
        .format(",.0~f"),
      vl.opacity()
        .fieldN("gesl"),
    )

  const textArrow = vl
    .markText({
      size: 14,
      dx: 30,
      dy: -16,
      angle: -90,
      align: "left",
      clip: true,
    })
    .encode(
      vl.y()
        .fieldQ("value"),
      vl.x()
        .fieldO("val"),
      vl.text()
        .value(" →"),
      vl.opacity()
        .fieldN("gesl"),
    )

    const layers = width > 600 ? 
      vl.layer(
        lines,
        points,
        text,
        textRel, 
        textGap,
        textArrow,
      ) :
      vl.layer(
        lines,
        points,
        textRel,
        textGap,
        textArrow,
        // text,
      )

  return layers
    .height(width > 600 ? 280 : 240)
    .width(inputWidth)
    .facet({
      field: "D", 
      type: "nominal", 
      title: null,
      // header: { 
      //   labelFontSize: 0, 
      // },
      // header: { 
      //   labelFontSize: 14, 
      //   orient: "bottom",
      // },
      header: { 
        labelFontSize: 14, 
        labelOrient: "top",
      },
    })
    .data(inputData)
    .config(config)

}

dumbbell(radio2 == "Beides anzeigen" ? decilesDumbbellFull : decilesDumbbellFull.filter(d => d.D == radio2), width > 600 ? 230 : width * 0.36).render({ renderer: "svg" }) 

viewof radio2 = Inputs.radio(
  ["Meine Vergleichsgruppe", "∅ Gesamtdurchschnitt", "Beides anzeigen"], 
  {value: "Meine Vergleichsgruppe", label: "Zeige"}
  ) 
```

```{ojs}
mehrWeniger = deciles.filter(d => d.D == d.closestDecileValue && d.val == "Status quo").map(d => d.gapValue)[0] > deciles.filter(d => d.D == d.closestDecileValue && d.val == "Szenario").map(d => d.gapValue)[0] ? " fallen." : " steigen."
mehrWenigerGesamt = deciles.filter(d => d.D == "Gesamt" && d.val == "Status quo").map(d => d.gapValue)[0] > deciles.filter(d => d.D == "Gesamt" && d.val == "Szenario").map(d => d.gapValue)[0] ? " sinken." : " steigen."

dumbbellText =  "Der Gender Pension Gap würde in meiner Vergleichsgruppe (" + deciles.map(d => d.closestDecileValue)[0] + " der " + radio + ") von " + germanLocale.format(",.0f")(deciles.filter(d => d.D == d.closestDecileValue && d.val == "Status quo").map(d => d.gapValue)[0]) + " Euro auf " + germanLocale.format(",.0f")(deciles.filter(d => d.D == d.closestDecileValue && d.val == "Szenario").map(d => d.gapValue)[0]) + " Euro" + mehrWeniger + " Damit würde die Frauenpension hier " + germanLocale.format(",.0%")(deciles.filter(d => d.D == d.closestDecileValue && d.val == "Szenario").map(d => d.gapRel)[0]) + " statt " + germanLocale.format(",.0%")(deciles.filter(d => d.D == d.closestDecileValue && d.val == "Status quo").map(d => d.gapRel)[0]) + " relativ zur Männerpension ausmachen. Insgesamt würde der Gap von " + germanLocale.format(",.0f")(deciles.filter(d => d.D == "Gesamt" && d.val == "Status quo").map(d => d.gapValue)[0]) + " Euro auf " + germanLocale.format(",.0f")(deciles.filter(d => d.D == "Gesamt" && d.val == "Szenario").map(d => d.gapValue)[0]) + " Euro" + mehrWenigerGesamt
```


${dumbbellText} <br><br> Eingaben ändern:


```{ojs}
Inputs.bind(Inputs.range([400, 4000], {step: 50, value: 1500, label: "Bruttopension"}), viewof rangeRaw)
Inputs.bind(Inputs.select(scenarios, {label: "Szenario"}), viewof select)
```


:::

::: {.slide}
## Das sind Ihre Ergebnisse
#### Ergebnisse 3/4
<span>Wie viel Pension erhalte ich momentan im Vergleich zu anderen?</span>


```{ojs}
distributions = {

  const dataDist = bins.filter(d => d.variable == "Status quo") //bins.filter(d => (d.variable == select || d.variable == "Status quo") && d.gesl == d.radio)
  // const dataRule = dataDist
  //   .filter(d => d.range == d.bin)
  //   .map(d => ({...d, y: 0, y2: 20000}))

const dataRule = [
      {gesl: "Frauen", median: 1223, y: 0, y2: 20000}, //1215
      {gesl: "Männer", median: 2286, y: 0, y2: 20000}, //2278
    ]

  const area = vl
    .markArea({ 
      interpolate: interpolate,
      fillOpacity: 0.4,
      strokeWidth: 1.75,
      clip: true,
    })
    .encode(
      vl.x()
        .fieldQ("bin")
        .axis({
          grid: false,
          title: "Bruttopension",
          // labels:  width > 825 ? true : false,
          // ticks:  width > 825 ? true : false,
        })
        .scale({
          domain: [0, 4500],
          nice: false,
        }),
      vl.y()
        .fieldQ("N")
        .scale({
          domain: [0, 23000],
          nice: false,
        })
        .axis({
          domain: false,
          grid: false,
          title: width > 825 ? "Anzahl Personen" : null,
          labels:  width > 825 ? true : false,
          ticks:  width > 825 ? true : false,
          values: [0, 5000, 10000, 15000, 20000],
        }),
      vl.color()
        .fieldN("gesl")
        .scale({
          range: colors2,
        })
        .legend(null),
        // .legend({
        //   orient: "top", 
        //   symbolOpacity: 1,
        //   symbolType: "square",
        //   symbolStrokeColor: "transparent",
        //   title: null,
        // }),
      vl.stroke()
        .fieldN("gesl")
        .scale({
          range: radio == "Männer" ? ["transparent", "black"] : ["black", "transparent"],
        })
        .legend(null),
    )

  const areaLowTail = vl
    .markArea({
      interpolate: interpolate,
      fillOpacity: 0.75,
      strokeWidth: 2,
      clip: true,
    })
    .transform(
      vl.filter("datum.bin <= datum.range"),
      vl.filter("datum.gesl == datum.radio"),
    )
    .encode(
      vl.x()
        .fieldQ("bin"),
      vl.y()
        .fieldQ("N"),
      vl.color()
        .fieldN("gesl")
        .scale({
          range: colors2,
        })
        .legend(null),
    )

  const rule = vl
    .markRule({
      stroke: "black",
      strokeDash: [4, 2],
      clip: true,
      strokeWidth: 0.75,
    })
    .transform(
      vl.filter("datum.bin == datum.range")
    )
    .encode(
      vl.x()
        .fieldQ("bin")
    )

  const ruleGray = vl
    .markRule({
      opacity: 0.51,
      clip: true,
    })
    .data(dataRule)
    .encode(
      vl.x()
        .fieldQ("median"),
      vl.y()
        .fieldQ("y"),
      vl.y2()
        .fieldQ("y2"),
      vl.opacity()
        .fieldN("gesl")
        .scale({
          range: radio == "Männer" ? 
            [0.51, 1] : 
            [1, 0.51]
        })
        .legend(null),
    )

  const textGray1 = vl
    .markText({
      opacity: 0.51,
      size: 13,
      y: 50,
      dx: -10,
      align: "right",
      clip: true,
    })
    .data(dataRule)
    .encode(
      vl.x()
        .fieldQ("median"),
      vl.text()
        .fieldQ("median")
        .format(",.0f"),
      vl.opacity()
        .fieldN("gesl")
        .scale({
          range: radio == "Männer" ? 
            [0.51, 1] : 
            [1, 0.51]
        })
        .legend(null),
    )

  const textGray2 = vl
    .markText({
      // opacity: 0.51,
      size: 11,
      y: 65,
      dx: -10,
      align: "right",
      clip: true,
    })
    .data(dataRule)
    .encode(
      vl.x()
        .fieldQ("median"),
      vl.text()
        .value("mittlere Pension"),
      vl.opacity()
        .fieldN("gesl")
        .scale({
          range: radio == "Männer" ? 
            [0.51, 1] : 
            [1, 0.51]
        })
        .legend(null),
    )

  const textGray3 = vl
    .markText({
      opacity: 0.51,
      size: 11,
      y: 80,
      dx: -10,
      align: "right",
      clip: true,
    })
    .data(dataRule)
    .encode(
      vl.x()
        .fieldQ("median"),
      vl.text()
        .fieldN("gesl"),
      vl.opacity()
        .fieldN("gesl")
        .scale({
          range: radio == "Männer" ? 
            [0.51, 1] : 
            [1, 0.51]
        })
        .legend(null),
    )

  const textLeft = vl
    .markText({
      clip: true,
      size: 14,
      y: 10,
      dx: -12,
      align: "right",
    })
    .transform(
      vl.filter("datum.gesl == datum.radio"),
      vl.filter("datum.bin == datum.range"),
    )
    .encode(
      vl.x()
        .fieldQ("bin"),
      vl.text()
        .value("← " + getCumPerc(radio, "lower")), 
    )

  // const textLeft2 = vl
  //   .markText({
  //     opacity: 0.51
  //     clip: true,
  //     size: 12,
  //     y: 42,
  //     dx: -12,
  //     align: "right",
  //   })
  //   .transform(
  //     vl.filter("datum.gesl == datum.radio"),
  //     vl.filter("datum.bin == datum.range"),
  //   )
  //   .encode(
  //     vl.x()
  //       .fieldQ("bin"),
  //     vl.text()
  //       .value("kleiner"), 
  //   )

  const textLeftOther = vl
    .markText({
      opacity: 0.51,
      clip: true,
      size: 12,
      y: 26,
      dx: -12,
      align: "right",
    })
    .transform(
      vl.filter("datum.gesl != datum.radio"),
      vl.filter("datum.bin == datum.range"),
    )
    .encode(
      vl.x()
        .fieldQ("bin"),
      vl.text()
        .value(getCumPerc(notRadio, "lower")), 
    )

  const textRight = vl
    .markText({
      clip: true,
      size: 14,
      y: 10,
      dx: 12,
      align: "left",
    })
    .transform(
      vl.filter("datum.gesl == datum.radio"),
      vl.filter("datum.bin == datum.range"),
    )
    .encode(
      vl.x()
        .fieldQ("bin"),
      vl.text()
        .value(getCumPerc(radio, "higher") + " →"), 
    )

  // const textRight2 = vl
  //   .markText({
  //     opacity: 0.51,
  //     clip: true,
  //     size: 12,
  //     y: 42,
  //     dx: 12,
  //     align: "left",
  //   })
  //   .transform(
  //     vl.filter("datum.gesl == datum.radio"),
  //     vl.filter("datum.bin == datum.range"),
  //   )
  //   .encode(
  //     vl.x()
  //       .fieldQ("bin"),
  //     vl.text()
  //       .value("größer"),
  //   )

  const textRightOther = vl
    .markText({
      opacity: 0.51,
      clip: true,
      size: 12,
      y: 26,
      dx: 12,
      align: "left",
    })
    .transform(
      vl.filter("datum.gesl != datum.radio"),
      vl.filter("datum.bin == datum.range"),
    )
    .encode(
      vl.x()
        .fieldQ("bin"),
      vl.text()
        .value(getCumPerc(notRadio, "higher")), 
    )

  const textPosition = vl
    .markText({
      clip: true,
      size: 18,
      y: 290,
      dx: 5,
      // angle: -0,
      align: "left",
      strokeWidth: 1.25,
    })
    .transform(
      vl.filter("datum.gesl == datum.radio"),
      vl.filter("datum.bin == datum.range"),
    )
    .encode(
      vl.x()
        .fieldQ("bin"),
      vl.text()
        .value("↙"), 
    )

  // const textPositionWhite = vl
  //   .markText({
  //     clip: true,
  //     size: 18,
  //     y: 290,
  //     dx: 5,
  //     // angle: -0,
  //     align: "left",
  //     stroke: "white",
  //     strokeWidth: 2.75,
  //     // fontWeight: "bold",
  //   })
  //   .transform(
  //     vl.filter("datum.gesl == datum.radio"),
  //     vl.filter("datum.bin == datum.range"),
  //   )
  //   .encode(
  //     vl.x()
  //       .fieldQ("bin"),
  //     vl.text()
  //       .value("↙"), 
  //   )

  const textMen = vl
    .markText({
      opacity: radio == "Männer" ? 1 : 0.51,
      clip: true,
      size: 11,
      y: width > 600 ? 200 : 105,
      align: "center",
    })
    .data([
      {bin: 3500}
    ])
    .encode(
      vl.x()
        .fieldQ("bin"),
      vl.text()
        .value("⤺ Pensionen der Männer"), 
    )

  const textWomen = vl
    .markText({
      opacity: radio == "Frauen" ? 1 : 0.51,
      clip: true,
      size: 11,
      y: width > 600 ? 110 : 35,
      align: "center",
    })
    .data([
      {bin: width > 600 ? 400 : 600}
    ])
    .encode(
      vl.x()
        .fieldQ("bin"),
      vl.text()
        .value(["Pensionen", "der Frauen ↷ "]), 
    )


    const layers = width > 825 ?
      vl.layer(
        ruleGray,
        textGray1,
        textGray2,
        textGray3,
        area,
        areaLowTail,
        rule,
        textLeft,
        textRight,
        // textLeft2,
        // textRight2,
        // textPositionWhite,
        textPosition,
        textLeftOther,
        textRightOther,
        textMen,
        textWomen,
      ):
      vl.layer( 
        area, 
        areaLowTail,
        rule,
        textLeft,
        textRight,
        textMen,
        textWomen,
      )

  return layers
    .height(width > 825 ? 300 : 200)
    .width(makeWidth(width))
    // .facet({
    //   field: "variable", 
    //   type: "nominal", 
    //   title: null,
    //   header: { 
    //     labelFontSize: 14, 
    //     orient: "top",
    //   },
    // })
    .data(dataDist)
    .config(config)
}

distributions.render({ renderer: "svg" }) 
textDistributions = "Etwa " + getCumPerc(radio, "lower") + " aller " + radio + " erhalten im Status quo eine niedrigere Pension als " + germanLocale.format(",.0f")(range) + " Euro pro Monat. Umgekehrt erhalten " +  getCumPerc(radio, "higher") + " aller " + radio + " mehr Pension."
```


${textDistributions} <br><br> Eingaben ändern:


```{ojs}
Inputs.bind(Inputs.range([400, 4000], {step: 50, value: 1500, label: "Bruttopension"}), viewof rangeRaw)
Inputs.bind(Inputs.radio(["Frau", "Mann"], {value: "Frauen", label: "Geschlecht"}), viewof radioRaw)
```


:::


::: {.slide}
## Das sind Ihre Ergebnisse
#### Ergebnisse 4/4 
<span>Wie würden sich die Verteilungen dieser Pensionen im ${select} ändern?</span>


```{ojs}
distributionsChange = {

  const dataDistChange = bins.filter(d => (d.variable == select || d.variable == "Status quo") && d.gesl == d.radio)
  const dataRule = dataDistChange
    .filter(d => d.range == d.bin && d.variable != "Status quo")
    .map(d => ({...d, y: 0, y2: 21500}))

  const colorsArea = radio == "Frauen" || select == "Szenario 2 (Höherbewertung Zeiten der Arbeitslosigkeit)" || select == "Szenario 5 (Beste 15 Jahre)"  ? colors : [colors[0], "transparent"]

  const area = vl
    .markArea({
      interpolate: interpolate,
      strokeWidth: 1.25,
      stroke: "black",
      opacity: 0.8,
      clip: true,
    })
    .encode(
      vl.x()
        .fieldQ("bin")
        .axis({
          grid: false,
          title: "Bruttopension",
          // labels:  width > 825 ? true : false,
          // ticks:  width > 825 ? true : false,
        })
        .scale({
          domain: [0, 4500],
          nice: false,
        }),
      vl.y()
        .fieldQ("N")
        .scale({
          domain: [0, 23000],
          nice: false,
        })
        .axis({
          domain: false,
          grid: false,
          title: width > 825 ? "Anzahl Personen" : null,
          labels:  width > 825 ? true : false,
          ticks:  width > 825 ? true : false,
          values: [0, 5000, 10000, 15000, 20000],
        }),
      vl.color()
        .fieldN("variable")
        .scale({
          range: colorsArea,
        })
        .legend(null)
        // .legend({
        //   orient: "top", 
        //   symbolType: "square",
        //   symbolOpacity: 1,
        //   symbolStrokeColor: "transparent", 
        //   labelLimit: width > 825 ? 1111 : 160,
        //   title: null,
        // })
    )

  const ruleGray = vl
    .markRule({
      // opacity: 0.51,
      clip: true,
    })
    .data(dataRule)
    .encode(
      vl.x()
        .fieldQ("median"),
      vl.y()
        .fieldQ("y"),
      vl.y2()
        .fieldQ("y2"),
    )

  const textGray1 = vl
    .markText({
      // opacity: 0.51,
      size: 13,
      y: 50,
      dx: -10,
      align: "right",
      clip: true,
    })
    .data(dataRule)
    .encode(
      vl.x()
        .fieldQ("median"),
      vl.text()
        .fieldQ("median")
        .format(",.0f"),
    )

  const textGrayVar = radio == "Frauen" || select == "Szenario 2 (Höherbewertung Zeiten der Arbeitslosigkeit)" || select == "Szenario 5 (Beste 15 Jahre)"  ? "neue mittlere Pension " + radio : "mittlere Pension " + radio 

  const textGray2 = vl
    .markText({
      // opacity: 0.51,
      size: 11,
      y: 65,
      dx: -10,
      align: "right",
      clip: true,
    })
    .data(dataRule)
    .encode(
      vl.x()
        .fieldQ("median"),
      vl.text()
        .value(textGrayVar),
    )

  const textGray3 = vl
    .markText({
      // opacity: 0.51,
      size: 11,
      y: 80,
      dx: -10,
      align: "right",
      clip: true,
    })
    .data(dataRule)
    .encode(
      vl.x()
        .fieldQ("median"),
      vl.text()
        .value(select.substr(0, 10)),
    )

  const textOld = vl
    .markText({
      clip: true,
      size: 14,
      y: 50,
      align: "center",
    })
    .data([
      {bin: radio == "Männer" ? 1000 : 600}
    ])
    .encode(
      vl.x()
        .fieldQ("bin"),
      vl.text()
        .value("alte Verteilung ⤵"), 
    )

  const textNewVar = radio == "Frauen" || select == "Szenario 2 (Höherbewertung Zeiten der Arbeitslosigkeit)" || select == "Szenario 5 (Beste 15 Jahre)"  ? "⤺ neue Pensionen" : "⤺ keine Veränderung"

  const textNew = vl
    .markText({
      // opacity: 0.51,
      clip: true,
      size: 11,
      y: width > 600 ? 180 : 100,
      align: "left",
    })
    .data([
      {bin: radio == "Männer" ? 2800 : 2200}
    ])
    .encode(
      vl.x()
        .fieldQ("bin"),
      vl.text()
        .value([textNewVar, select.substr(0, 10)]), 
    )

  const textArrow = vl
    .markText({
      // opacity: 0.51,
      clip: true,
      size: radio == "Frauen" || select == "Szenario 2 (Höherbewertung Zeiten der Arbeitslosigkeit)" || select == "Szenario 5 (Beste 15 Jahre)"  ? 16 : 0,
      y: 30,
      align: "right",
      opacity: 0.7,
      dx: -10,
    })
    .data(dataRule)
    .encode(
      vl.x()
        .fieldQ("median"),
      vl.text()
        .value("⟶"),
    )

  const layers = width > 825 ? 
    vl.layer(
      ruleGray,
      textGray1,
      textGray2,
      textGray3,
      textArrow,
      textNew,
      area, 
    ) :
    vl.layer(
      area, 
      textNew,
    )

  return layers
    .height(width > 825 ? 300 : 200)
    .width(makeWidth(width)) 
    .data(dataDistChange)
    .config(config)

}
distributionsChange.render({ renderer: "svg" })
```

```{ojs}
distributionsChangeText1 = 
  select == "Szenario 2 (Höherbewertung Zeiten der Arbeitslosigkeit)" ?
  "Durch die Aufwertung der Zeiten in Arbeitslosigkeit würden sich die Pension für " + radio + " leicht erhöhen." :
  radio == "Frauen" && select != "Szenario 2 (Höherbewertung Zeiten der Arbeitslosigkeit)" ?
  "Für " + radio + " würden sich die Pensionen im " + select + " deutlich erhöhen." :
  radio == "Männer" && select != "Szenario 5 (Beste 15 Jahre)" ?
  radio + " profitieren nicht von " + select + "." :
  radio + " profitieren deutlich von " + select + "."
distributionsChangeText2 = radio == "Männer" && select != "Szenario 5 (Beste 15 Jahre)" && select != "Szenario 2 (Höherbewertung Zeiten der Arbeitslosigkeit)" ? 
"Die mittlere Pension der Männer würde sich nicht verändern und bei " + germanLocale.format(",.0~f")(bins.filter(d => d.variable == select && d.gesl == d.radio)[0].median) + " verbleiben." :
"Die mittlere Pension der " + radio + " würde sich auf " + germanLocale.format(",.0~f")(bins.filter(d => d.variable == select && d.gesl == d.radio)[0].median) + " Euro steigern."
```


${distributionsChangeText1} ${distributionsChangeText2} <br><br> Eingaben ändern:


```{ojs}
Inputs.bind(Inputs.radio(["Frau", "Mann"], {value: "Frauen", label: "Geschlecht"}), viewof radioRaw) 
Inputs.bind(Inputs.select(scenarios, {label: "Szenario"}), viewof select)
```


:::

::: {.slide}

## Ergebnisse auf einen Blick
<!-- #### Ergebnisse auf einen Blick -->
<span>Hier können Sie die Ergebnisse gesammelt betrachten und Ihre Eingaben ändern</span>

<div class="fold-container">
  <div class="column">


      ```{ojs}
      Inputs.bind(Inputs.range([400, 4000], {step: 50, value: 1500, label: "Bruttopension"}), viewof rangeRaw) 
      Inputs.bind(Inputs.radio(["Frau", "Mann"], {value: "Frauen", label: "Geschlecht"}), viewof radioRaw)
      Inputs.bind(Inputs.select(scenarios, {label: "Szenario"}), viewof select)  
      ```

  </div>

  <div class="column">


      ```{ojs}
      Inputs.bind(Inputs.radio(["Meine Vergleichsgruppe", "∅ Gesamtdurchschnitt"], {value: "Meine Vergleichsgruppe", label: "Zeige"}), viewof radio2)
      Inputs.bind(Inputs.checkbox(["Kindererziehungszeiten"], {disabled: select == "Szenario 1 (Höherbewertung Kindererziehungszeiten)" && radio == "Frauen" ? false : true}), viewof check1) 
      Inputs.bind(Inputs.checkbox(["Arbeitslosenversicherungszeiten"], {disabled: select == "Szenario 2 (Höherbewertung Zeiten der Arbeitslosigkeit)" ? false : true}), viewof check2)
      ```

  </div>
</div>

<br>


```{ojs}
distributions.render({ renderer: "svg" })
```


:::: {.columns}

::: {.column width="50%"}

```{ojs}
bars(width > 600 ? 250 : width * 0.65).render({ renderer: "svg" })
```

:::

::: {.column width="50%"}

```{ojs}
dumbbell(radio2 == "Beides anzeigen" ? decilesDumbbellFull.filter(d => d.D == "Meine Vergleichsgruppe") : decilesDumbbellFull.filter(d => d.D == radio2), width > 600 ? 230 : width * 0.66).render({ renderer: "svg" })
```

:::


```{ojs}
distributionsChange.render({ renderer: "svg" })
```


::::

:::

<div class="btn-container">
<button id="prevBtn" class="btn btn-secondary">zurück</button>
<button id="nextBtn" class="btn btn-secondary">weiter</button>
<br>
<button id="jumpToLastBtn" class="btn btn-secondary small-btn">Überspringen und zur Vollversion (ohne Erläuterungen)</button>
</div>

::::

::: {.column-body .text-center}
<div class="btn-group" role="group" aria-label="Social Media Share Buttons">
  <a href="https://twitter.com/intent/tweet?text=AK%20Pensionsreformrechner&url=https://data-science.wifo.ac.at/AK-Pensionsreformrechner/" target="_blank" class="btn btn-secondary">
    <i class="fab fa-x-twitter"></i> 
  </a>
  <a href="https://www.facebook.com/sharer/sharer.php?u=https://data-science.wifo.ac.at/AK-Pensionsreformrechner/" target="_blank" class="btn btn-secondary">
    <i class="fab fa-facebook"></i> 
  </a>
  <a href="https://www.linkedin.com/shareArticle?mini=true&url=https://data-science.wifo.ac.at/AK-Pensionsreformrechner/" target="_blank" class="btn btn-secondary">
    <i class="fab fa-linkedin"></i> 
  </a>
  <a href="mailto:?subject=AK%20Pensionsreformrechner&body=https://data-science.wifo.ac.at/AK-Pensionsreformrechner/" target="_blank" class="btn btn-secondary">
    <i class="fas fa-envelope"></i> 
  </a>
  <a href="https://www.instagram.com/yourprofile/" target="_blank" class="btn btn-secondary">
    <i class="fab fa-instagram"></i> 
  </a>
</div>
:::


<!-- <iframe src="https://ooe.arbeiterkammer.at/newsletter" title="Newsletteranmeldung" width="100%"></iframe>  -->


```{html}
<form name="multiple_topics_subscription" method="post" action="https://example.com/newsletter">

    <div class="row art-box-white art-box-icon">
        <div class="col-md-4 nl-choose">
            <h2>Meine Newsletter</h2>
            <div class="fg-element">
                <div class="floating-label">
                    <div class="form-group">
                        <div id="multiple_topics_subscription_topics">
                            <div>
                                <div class="form-check">        
                                    <input type="checkbox" id="multiple_topics_subscription_topics_0" name="multiple_topics_subscription[topics][]" class="filled-in form-check-input" value="1002453">
                                    <label class="form-check-label" for="multiple_topics_subscription_topics_0">Arbeitswelt</label>
                                </div>
                            </div>
                            <!-- Repeat for other checkboxes -->
                        </div>
                    </div>
                </div>
            </div>
        </div>

        <div class="col-md-8">
            <h2>Meine Daten</h2>
            <div class="fg-element">
                <div class="floating-label">
                    <div class="form-group focused">
                        <label class="salutation control-label required" for="multiple_topics_subscription_salutation">Anrede <span class="mandatory">*</span></label>
                        <select id="multiple_topics_subscription_salutation" name="multiple_topics_subscription[salutation]" class="form-control" required>
                            <option value="0" selected>Bitte wählen</option>
                            <option value="1">Frau</option>
                            <option value="2">Herr</option>
                            <option value="3">Inter/Divers</option>
                            <option value="4">Keine Angabe</option>
                        </select>
                    </div>
                </div>
            </div>

            <div class="fg-element">
                <div class="floating-label">
                    <div class="form-group">
                        <label class="control-label required" for="multiple_topics_subscription_firstname">Vorname <span class="mandatory">*</span></label>
                        <input type="text" id="multiple_topics_subscription_firstname" name="multiple_topics_subscription[firstname]" required class="form-control">
                    </div>
                </div>
            </div>

            <div class="fg-element">
                <div class="floating-label">
                    <div class="form-group">
                        <label class="control-label required" for="multiple_topics_subscription_lastname">Nachname <span class="mandatory">*</span></label>
                        <input type="text" id="multiple_topics_subscription_lastname" name="multiple_topics_subscription[lastname]" required class="form-control">
                    </div>
                </div>
            </div>

            <div class="fg-element">
                <div class="floating-label">
                    <div class="form-group">
                        <label class="control-label required" for="multiple_topics_subscription_email">E-Mail <span class="mandatory">*</span></label>
                        <input type="email" id="multiple_topics_subscription_email" name="multiple_topics_subscription[email]" required class="form-control">
                    </div>
                </div>
            </div>

            <p><small><span class="mandatory">*</span> Pflichtfeld</small></p>
            <p><small>Ihre E-Mail-Adresse wird nicht an Dritte weitergegeben und zu keinem anderen Zweck verwendet...</small></p>

            <div class="form-group">
                <button type="submit" id="multiple_topics_subscription_subscribe" name="multiple_topics_subscription[subscribe]" class="btn-primary btn">Anmelden</button>
            </div>
        </div>
    </div>
</form>
```



<script>
  const slides = document.querySelectorAll('.slide');
  const prevBtn = document.getElementById('prevBtn');
  const nextBtn = document.getElementById('nextBtn');
  const conditionalInputs = document.getElementById('conditionalInputs');
  const topHeader = document.querySelector('.scroll-target'); // Get the header element
  let currentSlide = 0;

  // Function to handle slide transitions
  function showSlide(index) {
    // Hide the current slide with fade-out effect
    slides[currentSlide].classList.remove('fade-in');
    slides[currentSlide].classList.add('fade-out');

    // After fade-out, update to the new slide and show it with fade-in
    setTimeout(() => {
      slides[currentSlide].classList.remove('active');
      currentSlide = index;
      slides[currentSlide].classList.remove('fade-out');
      slides[currentSlide].classList.add('active', 'fade-in');

      // // Show or hide conditional inputs based on the slide number
      // if (currentSlide >= 9) { 
      //   conditionalInputs.classList.add('active');
      // } else {
      //   conditionalInputs.classList.remove('active');
      // }

      topHeader.scrollIntoView({ behavior: 'smooth' });

      // Update button states
      updateButtonStates();
    }, 300); // Adjust timing to match the fade-out duration
  }

  // Function to update the state of the buttons
  function updateButtonStates() {
    // Disable Previous button on the first slide
    if (currentSlide < 2) {
      prevBtn.disabled = true;
      prevBtn.classList.add('disabled');
      jumpToLastBtn.style.display = 'none'; // 'inline-block'
    } else {
      prevBtn.disabled = false;
      prevBtn.classList.remove('disabled');
      jumpToLastBtn.style.display = 'none'; // Hide "Jump to last page" button
    }

    // Change Next button text on the last slide
    if (currentSlide === slides.length - 1) {
      nextBtn.textContent = 'zum Anfang';
    } else if(currentSlide === 0) {
      nextBtn.textContent = 'weiter'; //  'Antwort bestätigen';
    } else if(currentSlide === 1) {
      nextBtn.textContent = 'Rechner starten';
    } else {
      nextBtn.textContent = 'weiter';
    }
  }

  // Attach event listeners to buttons
  prevBtn.addEventListener('click', () => {
    if (currentSlide > 0) {
      const newSlide = (currentSlide - 1 + slides.length) % slides.length;
      showSlide(newSlide);
    }
  });

  nextBtn.addEventListener('click', () => {
    const newSlide = (currentSlide + 1) % slides.length;
    showSlide(newSlide);
  });

  jumpToLastBtn.addEventListener('click', () => {
    showSlide(slides.length - 1);
  });

  // Initially show the first slide and update button states
  showSlide(currentSlide);
  updateButtonStates();
</script>